# AI Code Vulnerability Detection Engine - Model Training

This notebook trains a Machine Learning model (Random Forest) to safely detect vulnerabilities in Python source code using Natural Language Processing (TF-IDF vectorization).

## 1. Setup Environment
Make sure to upload your `dataset.csv` file to the Colab environment.

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn joblib

## 2. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Load the dataset
try:
    df = pd.read_csv('dataset.csv')
    print(f"Successfully loaded dataset with {len(df)} rows.")
except FileNotFoundError:
    print("ERROR: dataset.csv not found!")
    print("Please click the 'Folder' icon on the left in Colab and upload 'dataset.csv' before running this cell.")

# Show class distribution
print("\nClass Distribution:")
print(df['category_name'].value_counts())

## 3. Exploratory Data Analysis (EDA)
Let's see the balance of our classes visually.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='category_name')
plt.title('Distribution of Code Vulnerabilities in Dataset')
plt.xticks(rotation=45)
plt.show()

## 4. Feature Engineering (TF-IDF Vectorization)
We use TF-IDF to convert Python syntax strings into mathematical arrays. We use short n-grams (1,2) to capture combinations like `cursor.execute(f"SELECT`.

In [ ]:
X = df['code_snippet']
y = df['label']

# Split the data - 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Initialize and fit Vectorizer
# analyzer='word', ngram_range=(1,2) means it looks at single words and pairs of words
vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Extracted {X_train_vec.shape[1]} unique features (words/token combinations) from the code.")

## 5. Model Training
We are training a Random Forest. This algorithm is incredibly fast and highly explainable.

In [ ]:
# Initialize the Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model!
rf_model.fit(X_train_vec, y_train)
print("Model training complete!")

## 6. Model Evaluation
Let's test the AI on the 20% of data it has never seen.

In [ ]:
# Make predictions on the test set
y_pred = rf_model.predict(X_test_vec)

# Print Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")

# Print detailed classification report
# Map numeric labels back to category names for readable reports
target_names = ['Safe (0)', 'SQLi (1)', 'Cmd Inject (2)', 'Unsafe Eval (3)', 'XSS (4)']
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

## 7. Confusion Matrix
This shows us exactly where the AI gets confused.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted by AI')
plt.ylabel('Actual Label')
plt.title('Confusion Matrix')
plt.show()

## 8. Export Model for Scanner App
We must save the trained model AND the vectorizer so our Streamlit Dashboard can use them later.

In [ ]:
# Save as .pkl files
joblib.dump(rf_model, 'vulnerability_rf_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("Model architectures successfully saved! You can now download `vulnerability_rf_model.pkl` and `tfidf_vectorizer.pkl` from the Colab file explorer.")